# Wikibase Bot Account Test

**Project:** Linked Open Exhibition — NFDI4Culture / Hochschule Hannover (BIM-126-02)  
**Wikibase instance:** https://wikibase.wbworkshop.tibwiki.io/  
**AI attribution:** GitHub Copilot (Claude Sonnet 4.6)

Tests that your Wikibase bot account credentials are valid and that the account has sufficient permissions to read and write items.

---

## Prerequisites

### 1. Create a bot account

1. Log in to the Wikibase instance with your regular user account.
2. Go to **Special:BotPasswords** (e.g. https://wikibase.wbworkshop.tibwiki.io/wiki/Special:BotPasswords).
3. Enter a name for your bot (e.g. `TestBot`) and click **Create**.
4. Grant at minimum: **Edit existing pages**, **Create, edit, and move pages**.
5. Copy the generated credentials — they look like `YourUsername@TestBot` / `random-password`.

### 2. Create a `.env` file

Create a file called `.env` in the **root of the repository** (it is gitignored). Use `.env.example` as a template:

```
WB_URL=https://wikibase.wbworkshop.tibwiki.io
WB_USER=YourUsername@YourBotName
WB_PASSWORD=your-generated-bot-password
```

> **Never commit `.env` to version control.** The `.env` file is listed in `.gitignore`.

### 3. Install dependencies

Run once in your terminal (or use the cell below):

```bash
pip install wikibaseintegrator python-dotenv
```

In [1]:
# Install dependencies (run once if not already installed)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "wikibaseintegrator", "python-dotenv"])

0

## Step 1 — Load credentials from `.env`

In [4]:
import os
from dotenv import load_dotenv

# Load .env from repository root (two levels up from this notebook)
env_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "..", ".env")
load_dotenv(dotenv_path=env_path)

WB_URL  = os.getenv("WB_URL",  "https://wikibase.wbworkshop.tibwiki.io")
WB_USER = os.getenv("WB_USER")
WB_PASS = os.getenv("WB_PASSWORD")

if not WB_USER or not WB_PASS:
    raise EnvironmentError(
        "WB_USER and WB_PASSWORD must be set in your .env file. "
        "See the Prerequisites section above."
    )

print(f"Wikibase URL : {WB_URL}")
print(f"Bot user     : {WB_USER}")
print(f"Password     : {'*' * len(WB_PASS)}")

Wikibase URL : https://wikibase.wbworkshop.tibwiki.io
Bot user     : Simon@SimonBot
Password     : ********************************


## Step 2 — Configure `wikibaseintegrator` and log in

`wikibaseintegrator` needs to know the API endpoint and SPARQL endpoint for your Wikibase instance. The login uses the bot credentials from your `.env` file.

In [5]:
from wikibaseintegrator import WikibaseIntegrator, wbi_login
from wikibaseintegrator.wbi_config import config as wbi_config

# Point wikibaseintegrator at the project Wikibase instance
wbi_config["MEDIAWIKI_API_URL"]    = f"{WB_URL}/w/api.php"
wbi_config["SPARQL_ENDPOINT_URL"]  = f"{WB_URL}/query/sparql"
wbi_config["WIKIBASE_URL"]         = WB_URL

# Log in using BotPasswords credentials (Username@BotName format)
login_instance = wbi_login.Login(user=WB_USER, password=WB_PASS)
wbi = WikibaseIntegrator(login=login_instance)

print("Login successful.")

Login successful.


## Step 3 — Read test: fetch an item

Retrieve item `Q1` from the Wikibase instance to confirm the API is reachable and your credentials can read data.

In [7]:
item = wbi.item.get(entity_id="Q1")

print(f"Item ID    : {item.id}")
print(f"Labels     : {dict(item.labels.values)}")
print(f"Descriptions: {dict(item.descriptions.values)}")
print("\nRead test PASSED.")

Item ID    : Q1
Labels     : {'en': <LanguageValue @34d010 _LanguageValue__language='en' _LanguageValue__value='SanboxItem' _LanguageValue__removed=False>}
Descriptions: {'en': <LanguageValue @d4b150 _LanguageValue__language='en' _LanguageValue__value='A sample item for etsting Bot credentials' _LanguageValue__removed=False>}

Read test PASSED.


## Step 4 — Write test: create a temporary item

Creates a clearly labelled test item to confirm your bot account has write permissions. The new item's QID is printed so you can inspect or manually delete it afterwards. The clean-up cell below (Step 5) deletes it programmatically.

In [8]:
from wikibaseintegrator.models import Labels, Descriptions

new_item = wbi.item.new()
new_item.labels.set(language="en", value="BOT TEST ITEM — safe to delete")
new_item.descriptions.set(language="en", value="Temporary item created by bot account test notebook. Delete after verification.")

new_item.write(summary="Bot account test: creating temporary test item")

TEST_QID = new_item.id
print(f"Write test PASSED. Created item: {TEST_QID}")
print(f"View at: {WB_URL}/wiki/{TEST_QID}")

Write test PASSED. Created item: Q2
View at: https://wikibase.wbworkshop.tibwiki.io/wiki/Q2


## Step 5 — Clean up: delete the test item

Deletes the item created in Step 4. Requires the `delete` right to be granted to the bot account.  
If deletion fails, log in to the Wikibase wiki and delete the item manually via its page.

In [9]:
import requests

# Use the MediaWiki Action API to delete the test item
session = login_instance.session

# Step 1: get a delete token
token_resp = session.get(
    wbi_config["MEDIAWIKI_API_URL"],
    params={"action": "query", "meta": "tokens", "format": "json"}
)
token = token_resp.json()["query"]["tokens"]["csrftoken"]

# Step 2: delete the page
delete_resp = session.post(
    wbi_config["MEDIAWIKI_API_URL"],
    data={
        "action": "delete",
        "title": f"Item:{TEST_QID}",
        "reason": "Bot account test — removing temporary test item",
        "token": token,
        "format": "json"
    }
)
result = delete_resp.json()

if "delete" in result:
    print(f"Deleted {TEST_QID} successfully. Clean-up PASSED.")
else:
    print(f"Deletion response: {result}")
    print(f"If deletion failed, delete {WB_URL}/wiki/{TEST_QID} manually.")

Deleted Q2 successfully. Clean-up PASSED.


---

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `EnvironmentError: WB_USER and WB_PASSWORD must be set` | `.env` file missing or in wrong location | Place `.env` in the repository root directory |
| `Login failed` / `Incorrect username or password` | Wrong credential format | Use `Username@BotName` format, not your regular password |
| `Read test` raises `KeyError` or `HTTPError` | Wikibase instance unreachable | Check the `WB_URL` value and your network connection |
| `Write test` raises a permission error | Bot account lacks edit rights | Re-create the bot password with **Edit existing pages** and **Create, edit, and move pages** grants |
| `Deletion` returns an error | Bot account lacks delete rights | Grant the bot the **Delete pages** permission, or delete the item manually |

**Bot password permissions needed for this notebook:** Edit existing pages · Create, edit, and move pages · Delete pages (optional, for clean-up)

---

## Part 2 — Background: `.env` files and `.gitignore`

### What is a `.env` file?

A `.env` file is a plain-text file that stores **environment variables** — key/value pairs used to configure an application at runtime. Instead of hardcoding sensitive values like passwords or API URLs directly in code, you put them in `.env` and read them with a library (here, `python-dotenv`).

Example `.env`:
```
WB_URL=https://wikibase.wbworkshop.tibwiki.io
WB_USER=MyUsername@MyBot
WB_PASSWORD=abc123xyz
```

**Why use it?** It keeps secrets out of your source code and lets different users supply their own credentials without changing any code.

Further reading: [GeeksforGeeks — How to Create and Use .env Files in Python](https://www.geeksforgeeks.org/how-to-create-and-use-env-files-in-python/)

---

### What is `.gitignore`?

A `.gitignore` file tells Git **which files and folders to ignore** — i.e. never track or commit them. Any file listed in `.gitignore` stays only on your local machine.

The `.env` file must always be listed in `.gitignore` to prevent credentials from being accidentally published to GitHub.

Example `.gitignore` entry:
```
.env
```

A `.env.example` file (with placeholder values, no real secrets) *is* committed, so other users know which variables they need to fill in.

Further reading: [GeeksforGeeks — What is .gitignore and How to Use It?](https://www.geeksforgeeks.org/what-is-gitignore-and-how-to-use-it/)